In [5]:
%pip -q install datasets

from datasets import load_dataset

DATASET_ID = "SecureAI-SE/http-attack-requests"
DATASET_REVISION = "9ada662402e0620cc4e3bd32b2cc3c2025bb3804"
SPLITS = ("train", "validation", "test")

dataset = load_dataset(DATASET_ID, revision=DATASET_REVISION)

frames = {
    split: dataset[split].to_pandas()[["request", "label"]].copy()
    for split in SPLITS
}

for split, frame in frames.items():
    print(f"{split}: {len(frame):,} rows")


[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
train: 4,900 rows
validation: 700 rows
test: 1,400 rows


In [6]:
MAX_LENGTH = 256
PAD_ID = 0
UNK_ID = 1

train_characters = sorted(set("".join(frames["train"]["request"])))
char_to_id = {char: index for index, char in enumerate(train_characters, start=2)}

print(f"Vocabulary size including PAD and UNK: {len(char_to_id) + 2}")

Vocabulary size including PAD and UNK: 73


In [7]:
import numpy as np

def encode_requests(requests):
    encoded = np.full(
        (len(requests), MAX_LENGTH),
        PAD_ID,
        dtype=np.int64,
    )

    for row_index, text in enumerate(requests):
        character_ids = [
            char_to_id.get(char, UNK_ID)
            for char in text[:MAX_LENGTH]
        ]
        encoded[row_index, :len(character_ids)] = character_ids

    return encoded


X = {
    split: encode_requests(frames[split]["request"].tolist())
    for split in SPLITS
}

y = {
    split: frames[split]["label"].to_numpy(dtype=np.int64)
    for split in SPLITS
}

for split in SPLITS:
    assert X[split].shape == (len(y[split]), MAX_LENGTH)
    assert X[split].min() >= PAD_ID
    assert X[split].max() < len(char_to_id) + 2

    print(f"{split}: X={X[split].shape}, y={y[split].shape}")

train: X=(4900, 256), y=(4900,)
validation: X=(700, 256), y=(700,)
test: X=(1400, 256), y=(1400,)


In [8]:
import pandas as pd

label_names = dataset["train"].features["label"].names
audit_rows = []

for split in SPLITS:
    text_lengths = frames[split]["request"].str.len().to_numpy()
    encoded_lengths = np.minimum(text_lengths, MAX_LENGTH)

    assert X[split].dtype == np.int64
    assert y[split].dtype == np.int64
    assert np.all((y[split] >= 0) & (y[split] < len(label_names)))
    assert np.array_equal(
        (X[split] != PAD_ID).sum(axis=1),
        encoded_lengths,
    )

    audit_rows.append({
        "split": split,
        "rows": len(y[split]),
        "truncated_requests": int((text_lengths > MAX_LENGTH).sum()),
        "unknown_characters": int((X[split] == UNK_ID).sum()),
    })

audit = pd.DataFrame(audit_rows).set_index("split")
display(audit)

print("All preprocessing checks passed.")

,rows,truncated_requests,unknown_characters
split,,,
train,4900,227,0
validation,700,32,0
test,1400,70,0


All preprocessing checks passed.
